<a href="https://colab.research.google.com/github/RDGopal/IB9LQ-2026/blob/main/Retrieval_Augmented_Generation_(RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Retrieval Augmented Generation (RAG)
LLMs typically have the following limitations:

1. Knowledge Cutoff: Their knowledge is limited to the data they were trained on (often months or years old).
2. Hallucinations: They can confidently make up facts or generate plausible-sounding but incorrect information.
3. Lack of Domain-Specific/Private Knowledge: They don't know the content of your organization's information.

RAG is a technique to overcome these limitations by giving the LLM access to external, specific, and up-to-date information at the time of answering. The RAG setup consists of the following key steps.

1. Document Loading:
2. Text Splitting: Why split? Documents are long, LLM context windows are limited. We need to break large documents into smaller, manageable chunks.
3. Embedding: How do we find relevant chunks quickly? We convert text chunks into embeddings which capture the semantic meaning, so chunks with similar meaning have similar vectors.
4. Vector Storage: Where do we store the vectors for fast search? A Vector Database allows searching for vectors similar to a query vector very quickly.
5. Query Embedding: When a user asks a question, convert the query text into a vector using the same embedding model used for the documents.
6. Retrieval: Use the query vector to search the Vector Store for the most similar document vectors. Retrieve the top K (e.g., 2-5) corresponding text chunks.
7. Context Stuffing: Take the original user query and the retrieved text chunks. Combine them into a single prompt for the LLM. The prompt will look something like: "Here is some context: [Retrieved Chunk 1] [Retrieved Chunk 2] ... Based on this context, answer the following question: [User Query]".
8. Answer Generation: The LLM receives the prompt containing the specific context. It generates an answer based on and limited by the provided context. This significantly reduces hallucinations and ensures the answer is relevant to the external data.

##Load Packages
We are working with HuggingFace for embeddings; ChromaDB as a vector store; and Llama-Index as our orchestrator.

In [ ]:
# Install LlamaIndex core, readers, specific integrations, and Chroma DB
!pip install llama-index llama-index-core llama-index-readers-file pypdf -q
!pip install llama-index-embeddings-huggingface -q
!pip install llama-index-llms-gemini -q
!pip install chromadb llama-index-vector-stores-chroma -q

##Data Loading & Processing
We will create a directory (`docs`) where we will load all the documents.

In [ ]:
import os

# Create a directory to hold the uploaded document
os.makedirs("docs", exist_ok=True)

Now upload the file (the Student Handbook PDF) to this directory.

In [ ]:
from google.colab import files

uploaded = files.upload()

# Move the uploaded file into the 'docs' directory
for filename in uploaded.keys():
    file_path = os.path.join("docs", filename)
    os.rename(filename, file_path)
    print(f"\nMoved '{filename}' to {file_path}")

Now we can set up our LLM access. Please use your Gemini API keys, and HuggingFace API keys as a secret!!

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.gemini import Gemini
from google.colab import userdata
import os

# 1. Setup Embedding Model (Captures semantic meaning)
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")

# 2. Setup LLM (Generates the final answer)
# Ensure your Google API key is saved in Colab's Secrets (the key icon on the left)
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

Settings.llm = Gemini(model="models/gemini-2.5-flash")

Split the documents into chunks that are overlapping.

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

# Load the PDF from the directory
print("Loading documents...")
documents = SimpleDirectoryReader("docs").load_data()
print(f"Loaded {len(documents)} pages.")

# No-frills sentence splitting (basic chunking, no windowing)
# Nodes in LlamaIndex are equivalent to Chunks in LangChain
splitter = SentenceSplitter(chunk_size=200, chunk_overlap=25)
nodes = splitter.get_nodes_from_documents(documents)
print(f"Split document into {len(nodes)} nodes (chunks).")

# View an example chunk
if nodes:
    print(f"\n--- Example Chunk 1 ---")
    print(nodes[0].get_content())

##Embedding and Indexing

Create and populate the vector store

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

# 1. Initialize Chroma DB client and create a collection
# This creates a local directory named "chroma_db" to persist the vectors
print("Initialising Chroma DB...")
db = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = db.get_or_create_collection("rag_collection")

# 2. Set up the vector store and storage context for LlamaIndex
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 3. Create the Vector Store Index using the parsed nodes and Chroma
print("Creating Vector Store Index in Chroma DB...")
index = VectorStoreIndex(nodes, storage_context=storage_context)
print("Index created successfully.")

# 4. Configure the Query Engine
query_engine = index.as_query_engine(similarity_top_k=3)

# 5. Execute a Query
query = "Who is the Course Director?" #@param {type:"string"}
print(f"\nRunning Query: '{query}'\n")

response = query_engine.query(query)

print("--- Generated Answer ---")
print(response.response)

print("\n--- Source Chunks Used ---")
for i, node in enumerate(response.source_nodes):
    page = node.metadata.get('page_label', 'Unknown')
    print(f"\nSource {i+1} (Page {page}):")
    # Removed the [:200] slice to print the entire chunk
    print(node.get_content())
    print("-" * 50) # Added a separator for readability

#Your turn

##Task One
Create a set of test questions to verify that the RAG system is working as expected. This should be a mixture of easy, medium and hard questions (but questions that should be answerable based on the handbook). Test to see if the system works in all settings.

##Task Two
Play with the chunking strategy and top k hyperparameter. [This resource](https://developers.llamaindex.ai/python/framework/module_guides/loading/node_parsers/modules/) may be helpful.